# DART — Judge-winning Colab pipeline (baseline vs small LM vs Unsloth)

> **Newer end-to-end notebook (episode logs, multi-panel figures, self-repair):** open `training/colab_publication_pipeline.ipynb`.

Main judging notebook for end-to-end training evidence (bar charts + `train_reinforce_twin.py` CLI).

**What you compare (same evaluation protocol):**
- **Random policy** — cheap baseline (`RandomAgent` in the digital twin).
- **Small HF LM** — `distilgpt2` trained with **REINFORCE** on `DigitalTwinDiabetesEnv` rollouts.
- **Unsloth / large LM** — your chosen **~7–8B instruct** checkpoint (4-bit on GPU), same training script and **same held-out eval seeds** as the small run.

**Outputs for GitHub:**
- `logs/training_last_distil.json`, `logs/training_last_unsloth.json`
- `docs/figures/judge_bar_baseline_vs_small.png`
- `docs/figures/judge_bar_baseline_vs_unsloth.png`
- (Optional) learning-curve overlay: run `plot_compare_training_runs.py` as in README §7b

This replaces the generic “metaguard + HTTP server” template: here the **environment is in-process** (no separate `uvicorn` step required for training).

## STEP 1 — Clone repo & install
Edit `YOUR_REPO_URL` and `CLONE_DIR`.

**Two common layouts** (the next cell detects automatically):
- **Monorepo:** clone contains `DART/scripts/train_reinforce_twin.py` → we `cd` into `DART/`.
- **DART-only repo** (e.g. `github.com/.../DART`): `scripts/train_reinforce_twin.py` is at the **clone root** → we `cd` there (no nested `DART/`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from typing import Optional

YOUR_REPO_URL = "https://github.com/<YOUR_USERNAME>/<YOUR_REPO>.git"
# Target folder for `git clone` (created if missing). Example monorepo: .../mano ; DART-only: .../DART
CLONE_DIR = Path("/content/mano")

_marker = Path("scripts") / "train_reinforce_twin.py"


def _find_dart_root(base: Path) -> Optional[Path]:
    """Monorepo: base/DART/...  OR  DART-only repo: base/..."""
    nested = base / "DART" / _marker
    root = base / _marker
    if nested.is_file():
        return (base / "DART").resolve()
    if root.is_file():
        return base.resolve()
    return None


if not CLONE_DIR.is_dir():
    subprocess.run(["git", "clone", YOUR_REPO_URL, str(CLONE_DIR)], check=True)

REPO_ROOT = _find_dart_root(CLONE_DIR)
if REPO_ROOT is None and (CLONE_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)
    REPO_ROOT = _find_dart_root(CLONE_DIR)
if REPO_ROOT is None:
    raise FileNotFoundError(
        f"Could not find {_marker} under {CLONE_DIR} or {CLONE_DIR / 'DART'}.\n"
        "1) Open the notebook from GitHub (or pull latest) — old cells assumed a nested DART/ folder.\n"
        "2) Colab: Files → delete folder " + str(CLONE_DIR) + " OR set CLONE_DIR = Path('/content/dart_fresh') and rerun.\n"
        "3) YOUR_REPO_URL must be the DART repo (e.g. github.com/.../DART.git) or a monorepo that contains DART/."
    )

os.chdir(REPO_ROOT)
os.environ["DART_REPO_ROOT"] = str(REPO_ROOT.resolve())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "-r", "requirements_hackathon.txt"],
    check=True,
)
print("REPO_ROOT:", REPO_ROOT.resolve())

## STEP 2 — Enable GPU
Menu: **Runtime → Change runtime type → Hardware accelerator: T4 or A100** (A100 recommended for 8B + training).

## STEP 3 — Hugging Face token & Unsloth model id
- **HF_TOKEN**: required for **gated** models (e.g. Llama family). Create a read token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
- **`MODEL_ID`**: a **bnb-4bit** instruct model you can fit in VRAM (also copied to `MODEL_ID_UNSLOTH` for the training cell).

Edit the next cell: replace the placeholder with your **read** token (do not commit real tokens to GitHub).

In [ ]:
import os
from pathlib import Path

_HF_PLACEHOLDER = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
os.environ["HF_TOKEN"] = _HF_PLACEHOLDER  # in Colab: paste your real read token here (keep placeholder in git)

os.environ["MODEL_ID"] = "unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"
os.environ["MODEL_ID_UNSLOTH"] = os.environ["MODEL_ID"]

from huggingface_hub import login

if os.environ["HF_TOKEN"].strip() and os.environ["HF_TOKEN"] != _HF_PLACEHOLDER:
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

def _resolve_repo_root() -> Path:
    if os.environ.get("DART_REPO_ROOT"):
        return Path(os.environ["DART_REPO_ROOT"]).resolve()
    marker = Path("scripts") / "train_reinforce_twin.py"
    for cand in (Path("/content/mano/DART"), Path("/content/mano"), Path("/content/DART")):
        if (cand / marker).is_file():
            return cand.resolve()
    raise FileNotFoundError("Run STEP 1 first (sets DART_REPO_ROOT), or set DART_REPO_ROOT manually.")


REPO_ROOT = _resolve_repo_root()
os.environ["DART_REPO_ROOT"] = str(REPO_ROOT)
if not (REPO_ROOT / "scripts" / "train_reinforce_twin.py").is_file():
    raise FileNotFoundError(f"Invalid DART_REPO_ROOT: {REPO_ROOT}")
os.chdir(REPO_ROOT)
print("cwd:", REPO_ROOT)

## STEP 4 — “Dataset / env” sanity (DART)
There is **no** `generate_dataset.py` — the twin is the **simulator**. This checks imports and env wiring.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path(os.environ["DART_REPO_ROOT"])
subprocess.run([sys.executable, "scripts/run_sanity.py"], cwd=root, check=True)

## STEP 5 — No separate HTTP env server (for training)
`train_reinforce_twin.py` calls **`DigitalTwinDiabetesEnv.reset` / `step`** directly.  
*(Optional later: your OpenEnv FastAPI Space is for remote clients; not required for this notebook’s training loop.)*

## STEP 6 — Same evaluation = same `--judge-schedule` + default seeds
Both training commands below use **`--judge-schedule`** (120 updates, 32 eval seeds, etc.).  
If you **OOM** on the large model, add the **same** shorter flags to **both** commands, e.g. `--updates 60 --episodes-per-update 2`.

## STEP 7 — Train **small** LM (cheap baseline policy)
Writes `logs/training_last_distil.json` + per-run curves under `docs/figures/`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ["DART_REPO_ROOT"])

cmd = [
    sys.executable,
    "scripts/train_reinforce_twin.py",
    "--judge-schedule",
    "--model",
    "distilgpt2",
    "--out-json",
    "logs/training_last_distil.json",
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

## STEP 8 — Train **Unsloth / large** LM (4-bit, uses your GPU)
Uses **`transformers` + `--load-in-4bit`** (bitsandbytes). You do **not** need `import unsloth` for weight loading; add `pip install unsloth` only if you extend with Unsloth training APIs later.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ["DART_REPO_ROOT"])
mid = os.environ["MODEL_ID_UNSLOTH"]
cmd = [
    sys.executable,
    "scripts/train_reinforce_twin.py",
    "--judge-schedule",
    "--load-in-4bit",
    "--model",
    mid,
    "--out-json",
    "logs/training_last_unsloth.json",
]
if "llama" in mid.lower() or "meta-llama" in mid.lower():
    cmd.append("--trust-remote-code")
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

## STEP 9 — Numbers for judges + **two bar charts**
Prints means, then writes the two PNGs you should **commit** to the repo.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

REPO_ROOT = Path(os.environ["DART_REPO_ROOT"])

plot_cmd = [
    sys.executable,
    "scripts/plot_judge_two_bar_charts.py",
    "--distil-json",
    "logs/training_last_distil.json",
    "--unsloth-json",
    "logs/training_last_unsloth.json",
    "--small-bar-label",
    "DistilGPT2",
    "--large-bar-label",
    "Large LM",
]
subprocess.run(plot_cmd, cwd=REPO_ROOT, check=True)

p1 = REPO_ROOT / "docs" / "figures" / "judge_bar_baseline_vs_small.png"
p2 = REPO_ROOT / "docs" / "figures" / "judge_bar_baseline_vs_unsloth.png"
display(Image(filename=str(p1)))
display(Image(filename=str(p2)))

## STEP 10 — Optional: learning-curve overlay (both LMs + random)
Uncomment after both JSON files exist.

In [ ]:
# import subprocess, sys
# subprocess.run([
#     sys.executable, "scripts/plot_compare_training_runs.py",
#     "--run", "logs/training_last_distil.json:DistilGPT2",
#     "--run", "logs/training_last_unsloth.json:Large LM",
#     "--out", "docs/figures/compare_small_vs_unsloth_curves.png",
# ], cwd=REPO_ROOT, check=True)

## STEP 11 — Commit paths (GitHub)
```bash
git add logs/training_last_distil.json logs/training_last_unsloth.json \
  docs/figures/judge_bar_baseline_vs_small.png \
  docs/figures/judge_bar_baseline_vs_unsloth.png
git commit -m "Add judge Colab training logs and bar charts"
```

## What to tell judges (script)

> We compare a **random treatment policy**, a **small language model** (DistilGPT2), and a **larger instruction-tuned model** (4-bit), all trained with the **same REINFORCE loop** on the **same digital twin environment** and evaluated on the **same held-out random seeds**. Episode return is the **sum of weekly rewards** from the simulator. The bar charts show **mean ± std** on that shared evaluation.